# AWQ smoke test on Qwen3-1.7B

This notebook runs a small PyTorch-only reproduction of AWQ on a few Qwen3-1.7B linear layers. It is intentionally a smoke test: start with a tiny layer subset, verify the calibration/search path, then scale the target layer list.

AWQ reference: Lin et al., *AWQ: Activation-aware Weight Quantization for On-Device LLM Compression and Acceleration*, arXiv:2306.00978.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / 'awq').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from awq import collect_linear_inputs, temporary_awq
from weight_handles.metrics import compute_perplexity

In [ ]:
# Great Lakes local path. Replace with 'Qwen/Qwen3-1.7B' if running from Hugging Face.
MODEL_NAME = '/scratch/huterer_root/huterer0/jiamingp/models/qwen3-1b7'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()
next(model.parameters()).device

In [ ]:
calib_texts = [
    'The city archive kept letters from travelers who crossed the mountains during winter.',
    'def normalize(xs):\n    total = sum(xs)\n    return [x / total for x in xs if total != 0]',
    'If a sequence satisfies a_n = 2a_{n-1} + 1 and a_0 = 0, compute the first four terms.',
    'User: Can you summarize the meeting notes?\nAssistant:',
]

eval_texts = calib_texts[:2]

Start with only the first block's MLP projections. For a larger run, add attention projections or set `target_layers = None` to collect every linear layer.

In [ ]:
target_layers = [
    'model.layers.0.mlp.gate_proj',
    'model.layers.0.mlp.up_proj',
    'model.layers.0.mlp.down_proj',
]

buffers = collect_linear_inputs(
    model,
    tokenizer,
    calib_texts,
    layer_names=target_layers,
    batch_size=1,
    max_length=128,
    max_tokens_per_layer=256,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
{name: tuple(x.shape) for name, x in buffers.items()}

In [ ]:
baseline_ppl = compute_perplexity(
    model,
    tokenizer,
    eval_texts,
    batch_size=1,
    max_length=128,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
baseline_ppl

In [ ]:
with temporary_awq(
    model,
    buffers,
    layer_names=target_layers,
    n_bits=4,
    group_size=128,
    zero_point=True,
    n_grid=10,
) as awq_results:
    awq_ppl = compute_perplexity(
        model,
        tokenizer,
        eval_texts,
        batch_size=1,
        max_length=128,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )

awq_results, baseline_ppl, awq_ppl, awq_ppl - baseline_ppl

If the smoke test works, scale gradually:

1. increase `max_tokens_per_layer` to 512 or 1024,
2. include all MLP layers,
3. include attention projections,
4. evaluate on a larger held-out text set.